In [ ]:
# 구글드라이브 연동
from google.colab import drive
drive.mount('/gdrive', force_remount=True)

# 구글 드라이브 파일 확인
!ls '/gdrive/My Drive/temp/'

# 반복되는 드라이브 경로 변수화
drive_path = '/gdrive/My Drive/temp/'

Mounted at /gdrive
 매수종목1.txt		   little-char.png	   stock-data.csv
 매수종목2.txt		   ml_class.png		  'stock price.xlsx'
 adult.data		   pca_1.png		  'stock valuation.xlsx'
 auto-mpg.csv		   pca_2.png		   test.db.db
 bmi_500.csv		   pca_3.png		   test.txt
 citibike.csv		   Picture1.png		   ThoraricSurgery.csv
 매수종목.csv		   population_number.csv   titanic.xls
 data_mine.csv		   ram_price.csv	   tmdb_5000_credits.csv
 data_population.csv	   ratings_small.txt	   tmdb_5000_movies.csv
 data_studentlist_en.csv   ratings_test.txt	   Traffic_Accident_2017.csv
 demo.docx		   ratings_train.txt	   train.csv
 diabetes.csv		   sample1.pdf		   user_id_mean.csv
 example.docx		   sample2.pdf		   wine.csv
 height_weight.txt	   sample_merge.pdf	   주가데이터.xlsx
 HelloWorld.xlsx	   sample.xlsx		   남북한발전전력량.xlsx
 house_price.csv	   score.csv
 lenna.png		   sonar.csv


# 지도학습 실습

## 피마 인디언 당뇨병 예측

당뇨병 여부를 판단하는 머신러닝 예측 모델을 수립

- [데이터세트 다운로드](https://www.kaggle.com/uciml/pima-indians-diabetes-database)

- 다운받은 archive.zip 압축 파일에서 diabetes.csv 을 이용하거나 또는 수업자료 temp 폴더에서 diabetes.csv을 이용

- 모델 적용 후 성능 평가 지표를 확인 (오차행렬, 정확도, 정밀도, 재현율, F1스코어, AUC)

In [ ]:
# Python libraries
# Classic,data manipulation and linear algebra
import pandas as pd
import numpy as np

# Plots
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import plotly.offline as py
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
import plotly.tools as tls
import plotly.figure_factory as ff
py.init_notebook_mode(connected=True)
# import squarify

# Data processing, metrics and modeling
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import precision_score, recall_score, confusion_matrix,  roc_curve, precision_recall_curve, accuracy_score, roc_auc_score
import lightgbm as lgbm
from sklearn.ensemble import VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_curve,auc
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_predict
from yellowbrick.classifier import DiscriminationThreshold

# Stats
import scipy.stats as ss
from scipy import interp
from scipy.stats import randint as sp_randint
from scipy.stats import uniform as sp_uniform

# Time
from contextlib import contextmanager
@contextmanager
def timer(title):
    t0 = time.time()
    yield
    print("{} - done in {:.0f}s".format(title, time.time() - t0))

#ignore warning messages
import warnings
warnings.filterwarnings('ignore')

/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:144: FutureWarning:

The sklearn.metrics.classification module is  deprecated in version 0.22 and will be removed in version 0.24. The corresponding classes / functions should instead be imported from sklearn.metrics. Anything that cannot be imported from sklearn.metrics is now part of the private API.



In [ ]:
data = pd.read_csv('/content/diabetes.csv')

In [ ]:
display(data.info(),data.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


None

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 사용자 행동 인식 동작 예측

스마트폰 센서를 이용한 사용자 행동 인식 데이터 세트를 기반으로 어떠한 동작인지 예측하는 머신러닝 모델을 수립

- [UCI HAR Dataset.zip 데이터세트 다운로드](http://archive.ics.uci.edu/ml/machine-learning-databases/00240/)

- 다운받은 'UCI HAR Dataset.zip' 압축 파일을 이용하거나 또는 수업자료 temp 폴더에서 human_activity 폴더에 있는 자료를 이용

- 압축파일 폴더와 파일설명
```
/test             -> 테스트 용도의 데이터 세트
/train            -> 학습 용도의 데이터 세트
activity_labels.txt -> 동작 레이블 값에 대한 설명
features.txt      -> 특성 인덱스와 특성이름
features_info.txt -> 특성에 대한 간략 설명
README.txt        -> 데이터 세트에 대한 설명
```

- Pandas를 이용하여 데이터 분석 및 전처리
- GridSearch를 이용한 평균 정확도와 최적의 하이퍼파라미터를 확인


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

feature_name_df = pd.read_csv('/content/features.txt', sep='\s+', header=None, names=['column_index', 'column_name'])

feature_name_df

In [ ]:
feature_dup_df = feature_name_df.groupby('column_name').count()

feature_dup_df

In [ ]:
# 원본 데이터에 중복된 Feature 명으로 인하여 신규 버전의 Pandas에서 Duplicate name 에러를 발생.
# 중복 feature명에 대해서 원본 feature 명에 '_1(또는2)'를 추가로 부여하는 함수인 get_new_feature_name_df() 생성
def get_new_feature_name_df(old_feature_name_df):
  feature_dup_df = pd.DataFrame(data=old_feature_name_df.groupby('column_name').cumcount(),
                                columns=['dup_cnt'])
  feature_dup_df = feature_dup_df.reset_index()
  new_feature_name_df = pd.merge(old_feature_name_df.reset_index(), feature_dup_df, how='outer')
  new_feature_name_df['column_name'] = new_feature_name_df[['column_name', 'dup_cnt']].apply(lambda x : x[0]+'_'+str(x[1])
                                                                                        if x[1] >0 else x[0] ,  axis=1)
  new_feature_name_df = new_feature_name_df.drop(['index'], axis=1)
  return new_feature_name_df

In [ ]:
import pandas as pd

def get_human_dataset( ):

    # 각 데이터 파일들은 공백으로 분리되어 있으므로 read_csv에서 공백 문자를 sep으로 할당.
    feature_name_df = pd.read_csv(drive_path + 'human_activity/features.txt',sep='\s+',
                        header=None,names=['column_index','column_name'])

    # 중복된 피처명을 수정하는 get_new_feature_name_df()를 이용, 신규 피처명 DataFrame생성.
    new_feature_name_df = get_new_feature_name_df(feature_name_df)

    # DataFrame에 피처명을 컬럼으로 부여하기 위해 리스트 객체로 다시 변환
    feature_name = new_feature_name_df.iloc[:, 1].values.tolist()

    # 학습 피처 데이터 셋과 테스트 피처 데이터을 DataFrame으로 로딩. 컬럼명은 feature_name 적용
    X_train = pd.read_csv(drive_path + 'human_activity/train/X_train.txt',sep='\s+', names=feature_name )
    X_test = pd.read_csv(drive_path + 'human_activity/test/X_test.txt',sep='\s+', names=feature_name)

    # 학습 레이블과 테스트 레이블 데이터을 DataFrame으로 로딩하고 컬럼명은 action으로 부여
    y_train = pd.read_csv(drive_path + 'human_activity/train/y_train.txt',sep='\s+',header=None,names=['action'])
    y_test = pd.read_csv(drive_path + 'human_activity/test/y_test.txt',sep='\s+',header=None,names=['action'])

    # 로드된 학습/테스트용 DataFrame을 모두 반환
    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = get_human_dataset()

In [ ]:
print('###학습 피처 데이터셋 info()')
print(X_train.info())

In [ ]:
print(y_train['action'].value_count())

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt_clf.fit(X_train, y_train)
pred = df_clf.predict(X_Test)
accuracy = accuracy_score(y_test, pred)
print('결정 트리 예측 정확도:{0:.4f}'.format(accuracy))

print('DecisionTreeClassifier 기본 하이퍼 파라미터:\n', dt_clf.get_params())

In [ ]:
from sklearn.model_selection import GriidSearchCV

params = {
    'max_depth': [6,8,10,12,20,24]
}

grid_cv = GridSearchCV(dt_clf, param_grid=params, scoring = 'accuracy', cv=5, verbose =1)
grid_cv.fit(X_train, y_train)
print(format(grid_cv.best_score_))
print(grid_cv.best_params_)

In [ ]:
cv_results_df = pd.DataFrame(grid_cv.cv_results_)
cv_results_df[['param_max_depth', 'mean_test_score']]

In [ ]:
max_depths = [6,8,10,12,16,20,24]

for depth in max_depths:
  dt_clf = DecisionTreeClassifier(max_depth = depth, random_state=156)
  dt_clf.fit(X_train, y_train)
  pred = dt_clf.predict(X_test)
  accuracy = accuracy_score(y_test, pred)
  print(format(depth, accuracy))

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'max_depth': [6,8,10,12,20,24],
    'min_sample_split': [16,24]
}

grid_cv = GridSearchCV(dt_clf, param_grid=params, scoring='accuracy', cv=5, verbose=1)
print(format(grid_cv.best_score_))
print(grid_cv.best_params)

## 캐글 산탄데르 고객 만족 예측

370개 특성으로 주어진 데이터 세트 기반에서 고객 만족 여부를 예측하는 머신러닝 모델을 수립

- [데이터세트 다운로드](https://www.kaggle.com/c/santander-customer-satisfaction/data)

- 규칙준수 동의를 하고 Download All 버튼을 클릭해 압축 파일을 내려받은 후 train.csv를 추출하거나 왼쪽의 'Data Explorer'에서 train.csv 파일만 클릭해서 다운받은 후 이용

- 특성 이름은 모두 익명 처리돼 이름만을 가지고 어떤 속성인지 추정이 어려움. 클래스 레이블 명은 TARGET이며, 이 값이 1이면 불만, 0이면 만족한 고객

- 데이터 전처리 후 XGBoost와 LightGBM을 이용하여 모델 작성 및 ROC-AUC를 이용한 성능 평가와 GridSearch를 이용한 최적의 하이퍼파라미터를 확인

In [ ]:
import seaborn as sns

ftr_importances_values = best_df_clf.feature_importances_
ftr_importances = pd.Series(ftr_importances_values, index = X_train.columns)
ftr_top20 = ftr_importances.sort_values(ascending=False)[:20]
plt.title('Feature importances Top 20')
sns.barplot(x = ftr_top20, y=ftr_top20.index)
plt.show()

## 캐글 신용카드 사기 검출

매우 불균형한 분포를 가지고 있는 데이터 세트 기반에서 신용카드 사기 검출 머신러닝 모델을 수립

- [데이터세트 다운로드](https://www.kaggle.com/mlg-ulb/creditcardfraud)

- 규칙준수 동의를 하고 Download(144 MB) 버튼을 클릭해 압축 파일을 내려받은 후 creditcard.csv를 추출하거나 왼쪽의 'Data Explorer'에서 creditcard.csv 파일만 클릭해서 다운받은 후 이용

- 클래스 레이블 명은 Class 이며, 이 값이 1이면 사기 트랜잭션, 0이면 정상 트랜잭션

- 모델 적용 후 성능 평가 지표를 확인 (오차행렬, 정확도, 정밀도, 재현율, F1스코어, AUC)
